# App-32 — Szpiro : Pasten 2026 rend `N log log N` inconditionnel

[← Applications Search](../../README.md) | [Série Search](../../../README.md)

**Distillation d'un preprint.** Ce notebook extrait et rend *opératoire* le résultat de
Hector Pasten, *Improved Bounds for Szpiro's Conjecture* (arXiv 2609.17390, 16 septembre
2026) : le meilleur borne inconditionnelle sur la hauteur de Faltings d'une courbe
elliptique passe de `h(E) ≪ N log N` (Murty–Pasten, 2013) à `h(E) ≪ N log log N` — un
gain qui n'était connu auparavant que sous GRH (hypothèse de Riemann généralisée). En
outre, pour les courbes **semi-stables hors d'un ensemble fini de premiers**, la borne
devient `h(E) ≪ₛ N`.

Ce n'est pas un notebook de *recherche* : la preuve (courbes de Shimura, correspondance
de Jacquet–Langlands, multiplicité des valeurs propres de Hecke) est hors de portée
d'une session. C'est un notebook de **mécanique** : le théorème 1.1 énonce une borne
`h(E) ≪ M φ(D) log ℓ` indexée par une *factorisation admissible* `N = DM`, et ses deux
corollaires naissent du **choix optimal** de cette factorisation. Ce choix, lui, se
calcule — nous le calculons.

## 1. Le paysage conjectural

Pour une courbe elliptique `E` sur `ℚ`, on note `Δ` la valeur absolue du discriminant
minimal, `N` son conducteur, et `h(E)` sa hauteur de Faltings. Trois énoncés structurent
le sujet :

| Énoncé | Formule | Statut |
|---|---|---|
| Szpiro (conjecture) | `log Δ ≪ log N` | ouvert — implique une forme de *abc* |
| Frey (conjecture de hauteur) | `h(E) ≪ log N` | ouvert — **plus fort** que Szpiro |
| Lien (standard) | `log Δ ≪ max{1, h(E)}` | connu, effectif |

Le travail sérieux porte sur Frey. Avant 2026 : `h(E) ≪ N^(1/3+ε)` pour les courbes de
Frey, `h(E) ≪ N^(1/2+ε)` si `j(E)` est entier, et pour `E` quelconque le record datait de
2013 : `h(E) ≪ N log N` (Murty–Pasten). Sous GRH, Pasten avait obtenu
`h(E) ≪ N log log N`. **L'apport 2026 est de rendre cette borne inconditionnelle**, et
de donner `h(E) ≪ₛ N` dans le cas semi-stable hors `S`.

## 2. La définition-clé : factorisation admissible

> **Définition.** Une factorisation `N = DM` est **admissible** si `gcd(D, M) = 1`, si
> `D` est sans facteur carré, et si `D` a un **nombre pair** de facteurs premiers.

> **Théorème 1.1 (Pasten 2026).** Pour `E/ℚ`, soit `N = DM` une factorisation
> admissible et `ℓ` un premier ne divisant pas `N`. Alors
> `h(E) ≪ M·φ(D)·log ℓ`
> à constante absolue effective. *(Le preprint la déduit de la majoration
> `r_{D,M} ≪ φ(D)·M` du nombre de systèmes de valeurs propres de Hecke dans l'espace
> quaternionique `S_D(M)`.)*

La parité **paire** du nombre de facteurs premiers de `D` n'est pas décorative : elle
vient du signe de l'espace de formes quaternioniques `S_D(M)` dans lequel vit la newform
de `E` par Jacquet–Langlands. Une contrainte de parité si simple, si calculable —
voyons ce qu'elle fait.

In [1]:
from sympy import factorint, totient
from math import log, prod
from itertools import combinations

def admissible_factorisations(N):
    """Toutes les factorisations admissibles N = D*M (D carre-libre, nb PAIR de
    facteurs premiers, gcd(D, M) = 1). Retourne [(D, M)] triees par D croissant."""
    f = factorint(N)
    # gcd(D, N/D) = 1 avec D carre-libre => chaque premier de D apparait a l'exposant 1
    # exactement dans N (diviseur unitaire ET carre-libre).
    unitaires = [p for p, e in f.items() if e == 1]
    out = []
    for k in range(0, len(unitaires) + 1, 2):  # k PAIR : 0, 2, 4, ...
        for sel in combinations(unitaires, k):
            D = prod(sel)
            out.append((int(D), N // int(D)))
    return sorted(out)

def borne_11(N, D, M, ell):
    """Quantite M*phi(D)*log(ell) du theoreme 1.1 (a constante absolue pres).
    Identite utile : M*phi(D) = N * prod_{p | D} (1 - 1/p) <= N."""
    return M * int(totient(D)) * log(ell)

Sur `N = 30 = 2·3·5`, les trois premiers sont tous d'exposant 1 : chaque sous-ensemble
**pair** de `{2, 3, 5}` donne un `D` admissible. Il y en a exactement 4.

In [2]:
N = 30
rows = [(D, M, int(totient(D))) for D, M in admissible_factorisations(N)]
for D, M, phi in rows:
    print(f"D={D:2d}  M={M:2d}  phi(D)={phi:2d}   M*phi(D) = {M*phi:2d}")

D= 1  M=30  phi(D)= 1   M*phi(D) = 30
D= 6  M= 5  phi(D)= 2   M*phi(D) = 10
D=10  M= 3  phi(D)= 4   M*phi(D) = 12
D=15  M= 2  phi(D)= 8   M*phi(D) = 16


La lecture est immédiate : `D = 1` (la factorisation triviale) donne le coût
naïf `M·φ(D) = N = 30`, tandis que `D = 6 = 2·3` l'abaisse à `5·2 = 10` — via
l'identité `M·φ(D) = N·∏_{p|D}(1 − 1/p)` : mettre un premier dans `D` ne coûte que
le facteur `(1 − 1/p)` au lieu de le payer plein dans `M`. C'est tout le moteur du
théorème : la borne n'est pas `N` mais le *meilleur* `M·φ(D)` sur les factorisations admissibles.

In [3]:
def meilleure_borne(N):
    """Minimise M*phi(D) sur les factorisations admissibles de N."""
    cands = [(M * int(totient(D)), D, M) for D, M in admissible_factorisations(N)]
    return min(cands)

print(f"{'N':>6} {'nb fact. adm.':>13} {'D*':>6} {'M*':>6} {'M**phi(D*)':>10} {'vs N':>8}")
for N in [6, 30, 210, 778, 2310, 2 * 3 * 5 * 7 * 11 * 13]:
    val, D, M = meilleure_borne(N)
    n_adm = len(admissible_factorisations(N))
    print(f"{N:>6} {n_adm:>13} {D:>6} {M:>6} {val:>10d} {N/val:>7.1f}x")

     N nb fact. adm.     D*     M* M**phi(D*)     vs N
     6             2      6      1          2     3.0x
    30             4      6      5         10     3.0x
   210             8    210      1         48     4.4x
   778             2    778      1        388     2.0x
  2310            16    210     11        528     4.4x
 30030            32  30030      1       5760     5.2x


Deux régimes apparaissent dans le tableau :

- **N sans premiers répétés** (30, 210, 2310…) : le nombre de factorisations
  admissibles explose en `2^(ω(N)-1)` — la moitié des sous-ensembles pairs — et le
  meilleur `D` capte les petits premiers, là où `φ(D)/D` est le plus petit.
- **N avec premiers à puissance > 1** : ces premiers sont *exclus* de `D` (ils ne sont
  pas diviseurs unitaires carrés-libres) ; seul le noyau carré-libre d'exposant 1 compte.

C'est ce mécanisme combinatoire qui, poussé à l'échelle analytique (choix de `D` et du
premier `ℓ` optimisés simultanément), transforme le théorème 1.1 en corollaire 1.2 :
`h(E) ≪ N log log N` inconditionnel.

## 3. Côté courbes : que mesure la borne ?

Le théorème parle du **conducteur** `N` et de la hauteur de Faltings `h(E)` — deux
invariants profonds. Pour toucher du doigt l'*esprit* de Szpiro, nous prenons un proxy
pédagogique explicitement déclaré : pour `E: y² = x³ + ax + b` à coefficients entiers,
le discriminant du modèle vaut `Δ = -16(4a³ + 27b²)`, et nous comparons
`log|Δ|` à `log rad(|Δ|)` où `rad` est le radical (produit des premiers distincts).
**Ce n'est PAS l'énoncé de Szpiro** (qui met en jeu le conducteur et le discriminant
minimal) — c'est sa photographie de famille : la conjecture prédit `log Δ` contrôlé par
`log N`, et le radical est la quantité multiplicative la plus fine qui se calcule en
une ligne.

In [4]:
def delta_modele(a, b):
    """Discriminant (du modele) de y^2 = x^3 + a x + b."""
    return -16 * (4 * a**3 + 27 * b**2)

def rad(n):
    """Radical : produit des facteurs premiers distincts."""
    return prod(factorint(abs(n)).keys())

couples = [(1, 1), (0, 1), (2, 3), (1, 0), (3, 4)]
print(f"{'courbe':>16} {'|Delta|':>8} {'rad':>6} {'log|D|/log rad':>15}")
for a, b in couples:
    Dl = delta_modele(a, b)
    R = rad(Dl)
    if R > 1:
        q = log(abs(Dl)) / log(R)
        print(f"{'y2=x3+%dx+%d' % (a, b):>16} {abs(Dl):>8} {R:>6} {q:>15.3f}")

          courbe  |Delta|    rad  log|D|/log rad
      y2=x3+1x+1      496     62           1.504
      y2=x3+0x+1      432      6           3.387
      y2=x3+2x+3     4400    110           1.785
      y2=x3+1x+0       64      2           6.000
      y2=x3+3x+4     8640     30           2.665


Le ratio `log|Δ| / log rad(|Δ|)` oscille entre `1,5` et `6` sur cet échantillon :
plus le radical est petit devant `|Δ|`, plus la courbe est « extrémale » au sens de
Szpiro — c'est exactement le type de courbes (grandes hauteurs, petit conducteur) que
la conjecture de Frey dit impossibles au-delà d'un facteur constant, et que la borne
`N log log N` de 2026 rapproche du régime conjecturé de `log N`.

Sur ces cinq exemples minuscules, le ratio reste trivial ; les courbes extrémales
connues (celles qui approchent les records *abc*) ont des discriminants à dizaines de
chiffres — l'illustration reste qualitative, comme toute borne asymptotique sur
données finies.

## 4. Les corollaires, en prose fidèle

> **Corollaire 1.2.** Pour toute courbe elliptique `E/ℚ` : `h(E) ≪ N log log N`, à
> constante absolue **effective**. C'est la version inconditionnelle de ce qui
> exigeait GRH avant 2026.

> **Corollaire 1.3.** Soit `S` un ensemble fini de premiers. Pour toute `E/ℚ`
> **semi-stable hors `S`** : `h(E) ≪ₛ N` (constante dépendant seulement de `S`).

Le passage du théorème aux corollaires est le **choix de la factorisation** : pour
`Cor 1.2`, on équilibre `M` et `φ(D)` au fil des premiers de `N` (le `log log N`
résiduel est le prix du choix optimal en général) ; pour `Cor 1.3`, la semi-stabilité
force `N` carré-libre — **tous** ses premiers sont d'exposant 1, toutes les
factorisations admissibles sont disponibles, et le choix de `D` devient assez bon pour
tuer le `log log N` restant. Le caractère « admissible » n'est pas un artefact de
preuve : c'est l'interface entre la combinatoire de `N` et la parité des formes de
Shimura — dans la tradition de Serre, dont ce dépôt porte l'hommage pour les maths
pures.

## Exercice 1 — Énumérer les factorisations admissibles

Écrire `admissible_factorisations_verbose(N)` qui retourne, pour chaque factorisation
admissible `N = DM`, le tuple `(D, M, nb_premiers_de_D)`. La tester sur `N = 2·3·5·7`
et vérifier que le nombre de factorisations vaut `2^(ω(N)-1)` lorsque `N` est
carré-libre — et expliquer pourquoi la moitié seulement des sous-ensembles survit.

In [5]:
def admissible_factorisations_verbose(N):
    # TODO etudiant : reprendre admissible_factorisations et ajouter le nombre de
    # facteurs premiers de D a chaque ligne.
    pass

# Attendu sur N = 210 : 8 lignes, dont D=1 (0 premiers) et D=210 (4 premiers).
resultat = None  # TODO etudiant

## Exercice 2 — Le meilleur choix de `D`

Écrire `rapport_amelioration(N)` qui retourne le couple `(meilleur M·φ(D), N)` — la
quantité et le naïf qu'elle améliore. Sur la liste `N ∈ {30, 210, 2310, 30030}`,
calculer le gain `N / min` et observer où il est maximal. *Indice : le gain vient des
petits premiers à exposant 1 ; que se passe-t-il si on multiplie N par 2² ?*

In [6]:
def rapport_amelioration(N):
    # TODO etudiant : min de M*phi(D) sur les factorisations admissibles.
    pass

# Etape 1 : la table pour N dans {30, 210, 2310, 30030}
# Etape 2 : comparer avec 4*2310 (un premier repete) et expliquer la chute.
table = None  # TODO etudiant

## Exercice 3 — Cartographier les courbes extrémales

Écrire `top_ratios(paires, k)` qui, pour une liste de couples `(a, b)`, calcule le
ratio `log|Δ_modèle| / log rad(|Δ_modèle|)` et retourne les `k` plus grands. La tester
sur les couples `(a, b)` avec `0 ≤ a, b ≤ 6`, `4a³ + 27b² ≠ 0`. *Indice : les courbes
avec `4a³ + 27b²` à petit radical (puissances pures de 2 ou 3) dominent — pourquoi ?*

In [7]:
def top_ratios(paires, k=3):
    # TODO etudiant : ratios log|Delta|/log rad tries decroissants, top k.
    pass

# Etape 1 : enumerer les couples 0 <= a, b <= 6 avec discriminant non nul
# Etape 2 : afficher le top 3 et verifier a la main les deux premiers ratios
classement = None  # TODO etudiant

## Conclusion

- **Ce que le papier démontre** : théorème 1.1 (`h(E) ≪ M·φ(D)·log ℓ` pour toute
  factorisation admissible), corollaire 1.2 (`N log log N` **inconditionnel** — le
  record 2013 `N log N` tombe après 13 ans), corollaire 1.3 (`≪ₛ N` semi-stable
  hors `S`).
- **Ce que ce notebook calcule** : la mécanique du choix de `D` — le levier
  combinatoire qui transforme le théorème en corollaires — et le proxy
  `log|Δ|/log rad(|Δ|)` pour *sentir* ce que « contrôler la hauteur par la structure
  multiplicative du conducteur » veut dire.
- **Ce qui reste hors scope** : la preuve (courbes de Shimura, Jacquet–Langlands,
  lemme de multiplicité `µ_{D,M}(f, ℓ) ≪ φ(D)M log ℓ / log N`), le calcul du
  conducteur et du discriminant minimal (tables de Cremona/LMFDB), et l'énoncé Lean
  formel du théorème — objet d'un grain séparé pour une lane à capacité Lean.

**Source** : Hector Pasten, *Improved Bounds for Szpiro's Conjecture*, arXiv
2609.17390 (preprint, 2026-09-16), 5 pages. PDF archivé dans la bibliographie du dépôt
(`Bibliographie IA/NumberTheory`). Distillation issue de #16549 (scan de résonance
arXiv, hommage Serre pour les maths pures).